# 01 - Visão Geral do Projeto

## Descrição

Este projeto implementa um **pipeline completo de engenharia de dados** para um cenário de e-commerce fictício. A solução extrai dados de uma origem NoSQL (MongoDB), processa-os através de uma arquitetura medalhão (Landing → Bronze → Silver → Gold) sobre um Data Lake com object storage (MinIO/S3), e entrega um modelo dimensional analítico pronto para consumo em dashboards e ferramentas de BI.

A orquestração é feita pelo **Apache Airflow**, que gerencia quatro DAGs interdependentes com agendamento periódico. O processamento utiliza **Apache Spark** com **Delta Lake** para garantir transações ACID, time-travel e MERGE incremental.

---

## Problema de Negócio

### O Desafio

Uma empresa de e-commerce opera com um sistema transacional baseado em **MongoDB** que armazena pedidos, clientes, produtos, pagamentos, entregas e avaliações. Com o crescimento do volume de dados e a necessidade de tomada de decisão baseada em dados, a empresa precisa:

1. **Centralizar** dados dispersos em um Data Lake analítico
2. **Garantir qualidade** dos dados através de validação e limpeza
3. **Preservar histórico** de alterações (ex: mudança de endereço de cliente, preço de produto)
4. **Entregar métricas** como vendas por período, aprovação de pagamentos, prazo de entrega e satisfação do cliente
5. **Permitir análise temporal** com consistência de dados históricos

### Por que MongoDB como Origem?

O MongoDB foi escolhido deliberadamente para simular um cenário realista de sistemas modernos:

- **Schema flexível**: documentos JSON/BSON são nativos de muitas aplicações web
- **Heterogeneidade de tipos**: datas como `ISODate`, números como `int`/`double`, flags como `bool`
- **Inconsistências esperadas**: em uma base operacional real, não há garantia perfeita de integridade referencial ou consistência aritmética
- **Referência em vez de embedding**: as coleções são ligadas por IDs inteiros (não `ObjectId`), facilitando o mapeamento para modelo dimensional

> **Referência**: `docs/modelo_mongodb.md` — documentação completa da modelagem da origem.

---

## Casos de Uso Principais

### UC-01: Carga Incremental de Dados Operacionais

- **Ator**: Sistema (DAG `mongodb_to_landing`)
- **Descrição**: Extrair dados alterados do MongoDB desde a última execução e gravar na Landing
- **Frequência**: A cada 15 minutos
- **Resultado**: Arquivos JSON estendido no Data Lake com metadados de extração

### UC-02: Persistência em Formato Analítico

- **Ator**: Sistema (DAG `landing_to_bronze`)
- **Descrição**: Converter JSONs da Landing para tabelas Delta Lake na Bronze com auditoria
- **Frequência**: A cada 15 minutos (offset 5 min)
- **Resultado**: Tabelas Delta particionadas por data de ingestão, com metadados de linhagem

### UC-03: Limpeza e Conformação de Dados

- **Ator**: Sistema (DAG `bronze_to_silver`)
- **Descrição**: Aplicar regras de qualidade: deduplicação, validação de domínio, integridade referencial
- **Frequência**: A cada 15 minutos (offset 10 min)
- **Resultado**: Dados limpos e validados em Delta Lake, com log de rejeições

### UC-04: Modelagem Dimensional Analítica

- **Ator**: Sistema (DAG `silver_to_gold`)
- **Descrição**: Materializar dimensões SCD Tipo 2 e fatos com métricas de negócio
- **Frequência**: A cada 15 minutos (offset 15 min)
- **Resultado**: Modelo dimensional pronto para BI: 4 dimensões + 4 fatos

### UC-05: Análise de Vendas e Performance

- **Ator**: Analista de Negócio
- **Descrição**: Consultar fatos de vendas (`fato_vendas`) e dimensões de tempo, cliente, produto e cupom
- **Métricas**: Receita bruta, receita líquida, desconto aplicado, quantidade de itens

### UC-06: Análise de Pagamentos e Aprovação

- **Ator**: Analista Financeiro
- **Descrição**: Consultar fatos de pagamentos (`fato_pagamentos`) por período, cliente e forma de pagamento
- **Métricas**: Valor aprovado, taxa de aprovação, distribuição por forma de pagamento

### UC-07: Análise de Logística e Entregas

- **Ator**: Gestor de Operações
- **Descrição**: Consultar fatos de entregas (`fato_entregas`) por período e transportadora
- **Métricas**: Prazo real vs. previsto, atraso em dias, percentual de entregas no prazo

### UC-08: Análise de Satisfação do Cliente

- **Ator**: Gestor de Experiência do Cliente
- **Descrição**: Consultar fatos de avaliações (`fato_avaliacoes`) por produto e período
- **Métricas**: Nota média, percentual de avaliações positivas (nota ≥ 4), distribuição por estrela

---

## Stack Tecnológico

### Orquestração e Workflow

| Componente | Versão | Função |
|-----------|--------|--------|
| **Apache Airflow** | 3.2.2 | Orquestração de DAGs, agendamento, dependências, retentativa |
| **Python** | ≥ 3.11 | Linguagem de implementação (DAGs, jobs Spark, scripts) |

### Banco de Dados e Storage

| Componente | Versão | Função |
|-----------|--------|--------|
| **MongoDB** | 7 | Banco de origem NoSQL com dados transacionais do e-commerce |
| **MinIO** | RELEASE.2025-09-07 | Object storage S3-compatible para Data Lake local |
| **PostgreSQL** | 16 | Metadados do Airflow (backend do scheduler e webserver) |
| **Delta Lake** | 3.3.1 | Format de storage ACID sobre object storage (Bronze, Silver, Gold) |

### Processamento e Transformação

| Componente | Versão | Função |
|-----------|--------|--------|
| **Apache Spark** | 3.5.3 | Processamento distribuído (PySpark) entre camadas |
| **PySpark** | 3.5.3 | API Python para implementação dos jobs de transformação |

### Infraestrutura e DevOps

| Componente | Versão | Função |
|-----------|--------|--------|
| **Docker** | — | Containerização de todos os serviços (MongoDB, MinIO, Airflow, Postgres) |
| **Docker Compose** | — | Orquestração multi-container para ambiente local |
| **GitHub Actions** | — | CI/CD automático (testes e deploy de documentação) |
| **MkDocs Material** | ≥ 9.5 | Geração de site de documentação estático |

### Bibliotecas Python (Dependências)

| Grupo | Bibliotecas | Função |
|-------|------------|--------|
| `dataset` | pymongo, dnspython, faker, pandas | Geração e carga de dados sintéticos no MongoDB |
| `infra` | boto3 | Operações em S3/MinIO (criação de estruturas) |
| `spark` | delta-spark, pyspark | Jobs de transformação Delta Lake |
| `airflow` | apache-airflow-providers-amazon, apache-airflow-providers-apache-spark, apache-airflow-providers-mongo | Providers para conexões Airflow |
| `docs` | mkdocs-material, pymdown-extensions | Construção da documentação |

---

## Arquitetura de Alto Nível

### Visão Macro

O sistema opera em quatro camadas principais, com uma origem NoSQL e um consumo analítico:

```mermaid
graph LR
    subgraph Origem
        M["MongoDB Atlas<br/>10 coleções<br/>15K docs cada"]
    end

    subgraph Orquestração
        AF["Apache Airflow<br/>4 DAGs<br/>Agendamento 15min"]
    end

    subgraph Data Lake
        L["Landing<br/>JSON bruto<br/>cópia fiel da origem"]
        B["Bronze<br/>Delta Lake<br/>metadados de auditoria"]
        S["Silver<br/>Delta Lake<br/>limpo e validado"]
        G["Gold<br/>Delta Lake<br/>modelo dimensional"]
    end

    subgraph Consumo
        D["Dashboard / BI<br/>4 KPIs + 2 métricas"]
    end

    M -->|mongodb_to_landing| L
    L -->|landing_to_bronze| B
    B -->|bronze_to_silver| S
    S -->|silver_to_gold| G
    G --> D

    AF -.->|Agendamento| M
    AF -.->|Execução| L
    AF -.->|Execução| B
    AF -.->|Execução| S
    AF -.->|Execução| G

    style M fill:#15803d,color:#fff,stroke:#166534
    style L fill:#e2e8f0,stroke:#94a3b8,color:#1e293b
    style B fill:#fdba74,stroke:#b45309,color:#7c2d12
    style S fill:#cbd5e1,stroke:#64748b,color:#1e293b
    style G fill:#fde047,stroke:#a16207,color:#713f12
    style D fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
    style AF fill:#bfdbfe,stroke:#3b82f6,color:#1e3a8a
```

### Responsabilidades das Camadas

| Camada | Responsabilidade | Formato | Diferencial |
|--------|-----------------|---------|-------------|
| **Landing** | Cópia fiel da origem sem transformação | JSON estendido (MongoDB Extended JSON) | Preserva tipos BSON, permite reprocessamento |
| **Bronze** | Persistência em formato analítico com metadados | Delta Lake | ACID, schema enforcement, time-travel, particionamento por `ingestion_date` |
| **Silver** | Limpeza, tipagem, deduplicação, validação de qualidade | Delta Lake | Regras de negócio, integridade referencial, log de rejeições |
| **Gold** | Modelo dimensional analítico com histórico | Delta Lake | Dimensões SCD Tipo 2, fatos particionados por `ano`, surrogate keys |

---

## Contexto do Sistema (Diagrama C4 — Contexto)

```mermaid
graph TB
    subgraph Usuários
        AN[Analista de Negócio]
        PM[Product Manager]
        EXEC[Executivo]
    end

    subgraph Sistema - Pipeline de Dados
        AF[Apache Airflow]
    end

    subgraph Sistemas Externos
        M[(MongoDB<br/>Origem transacional)]
        MINIO[(MinIO<br/>Data Lake)]
        DASH[Dashboard / BI]
    end

    AN -->|Consulta métricas| DASH
    PM -->|Análise de tendências| DASH
    EXEC -->|Relatórios estratégicos| DASH

    AF -->|Extrai dados<br/>incrementais| M
    AF -->|Grava / processa| MINIO

    MINIO -->|Alimenta| DASH
    
    style M fill:#15803d,color:#fff
    style MINIO fill:#fbbf24,color:#713f12
    style DASH fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
    style AF fill:#bfdbfe,stroke:#3b82f6,color:#1e3a8a
```

---

## Componentes Principais

### 1. Origem de Dados (MongoDB)

- **Banco**: `ecommerce`
- **Coleções**: 10 (clientes, categorias, fornecedores, produtos, cupons, pedidos, itens_pedido, pagamentos, entregas, avaliacoes)
- **Volume**: ~15.000 documentos por coleção
- **Dados**: Sintéticos, gerados com Faker (locale `pt_BR`), determinísticos (sementes fixas)
- **Deploy**: Local via Docker (`localhost:27017`) ou Atlas compartilhado

> **Referência**: `docker-compose.yml` (serviço `mongodb`), `dataset/scripts_py/carregar_mongo.py`

### 2. Orquestração (Apache Airflow)

- **Versão**: 3.2.2
- **Executor**: LocalExecutor (single-node, adequado para desenvolvimento e demonstração)
- **DAGs**: 4 pipelines principais com agendamento escalonado (0, 5, 10, 15 min de cada ciclo de 15 min)
- **Interface**: Web UI em `http://localhost:8080` (usuário: `airflow`, senha: `airflow`)
- **Backend**: PostgreSQL 16 para metadados

> **Referência**: `dags/mongodb_to_landing.py`, `dags/landing_to_bronze.py`, `dags/bronze_to_silver.py`, `dags/silver_to_gold.py`

### 3. Data Lake (MinIO — S3-Compatible)

- **Bucket**: `datalake`
- **API**: S3 em `http://localhost:9000` (console: `http://localhost:9001`)
- **Versionamento**: Habilitado por padrão para auditoria e recuperação
- **Estrutura**: Prefixos por camada e tabela (`landing/ecommerce/...`, `bronze/ecommerce/...`, etc.)

> **Referência**: `scripts/criar_estrutura_landing.py`, `scripts/criar_estrutura_bronze.py`, etc.

### 4. Jobs de Transformação (PySpark + Delta Lake)

- **Motor**: Apache Spark 3.5.3
- **Formato**: Delta Lake 3.3.1 (ACID, MERGE, time-travel, schema enforcement)
- **Storage**: Acesso via S3A (S3-compatible) sobre MinIO
- **Pacotes**: `io.delta:delta-spark_2.12:3.3.1`, `org.apache.hadoop:hadoop-aws:3.3.4`

> **Referência**: `spark_jobs/landing_to_bronze.py`, `spark_jobs/bronze_to_silver.py`, `spark_jobs/silver_to_gold.py`

### 5. Modelagem Dimensional (Gold)

- **Dimensões**: `dim_tempo`, `dim_cliente` (SCD2), `dim_produto` (SCD2), `dim_cupom` (SCD2)
- **Fatos**: `fato_vendas`, `fato_pagamentos`, `fato_entregas`, `fato_avaliacoes`
- **Particionamento**: Fatos particionados por `ano` para otimização de consultas
- **SCD Tipo 2**: Preserva todas as versões históricas de atributos de cliente, produto e cupom

> **Referência**: `dags/lib/silver_gold.py` (GOLD_MODELS), `config/gold_structure.json`

---

## Produtos e Entregáveis do Sistema

### Produto 1: Pipeline de Dados ETL/ELT
- **Descrição**: Sistema automatizado que extrai, transforma e carrega dados do MongoDB para um modelo dimensional analítico
- **SLA**: Carga completa em ~15 minutos (ciclo de 4 DAGs)
- **Garantias**: Idempotência, checkpoint incremental, qualidade de dados, auditoria completa

### Produto 2: Data Lake Analítico
- **Descrição**: Repositório centralizado com 4 camadas de dados, versionado e auditável
- **Formato**: Delta Lake sobre S3-compatible (MinIO local / Amazon S3 em produção)
- **Benefícios**: Time-travel, schema enforcement, MERGE incremental, particionamento inteligente

### Produto 3: Modelo Dimensional para BI
- **Descrição**: 4 dimensões + 4 fatos com métricas de negócio prontas para consumo
- **Métricas**: Vendas, pagamentos, entregas, avaliações com indicadores de performance
- **Histórico**: SCD Tipo 2 permite análise temporal consistente ("como o cliente era na época da compra")

### Produto 4: Documentação Técnica Publicada
- **Descrição**: Site estático gerado via MkDocs Material, publicado automaticamente no GitHub Pages
- **URL**: `https://olucasoliverio.github.io/Engenharia_Dados_Final/`
- **Conteúdo**: Arquitetura, modelos de dados, guias de operação, referências

---

## Fluxo de Negócio de Alto Nível

```mermaid
flowchart TD
    A[Sistema Operacional<br/>E-commerce] -->|Transações<br/>Pedidos, Clientes, Pagamentos| B[(MongoDB)]
    B -->|DAG 1: Extração<br/>Incremental| C[Landing<br/>JSON Estendido]
    C -->|DAG 2: Conversão<br/>Delta Lake| D[Bronze<br/>Tabelas Delta]
    D -->|DAG 3: Limpeza<br/>Validação| E[Silver<br/>Dados Confiáveis]
    E -->|DAG 4: Modelagem<br/>Dimensional| F[Gold<br/>Dimensões + Fatos]
    F -->|Consultas<br/>OLAP| G[Dashboards<br/>Relatórios]
    G -->|Decisões| H[Gestão de Negócio]

    style A fill:#dbeafe,stroke:#3b82f6,color:#1e3a8a
    style B fill:#15803d,color:#fff,stroke:#166534
    style C fill:#e2e8f0,stroke:#94a3b8,color:#1e293b
    style D fill:#fdba74,stroke:#b45309,color:#7c2d12
    style E fill:#cbd5e1,stroke:#64748b,color:#1e293b
    style F fill:#fde047,stroke:#a16207,color:#713f12
    style G fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
    style H fill:#fce7f3,stroke:#db2777,color:#831843
```

---

## Decisões de Arquitetura Principais

| Decisão | Escolha | Justificativa |
|---------|---------|--------------|
| Origem de dados | MongoDB (NoSQL) | Simula cenário realista de dados semiestruturados e inconsistentes |
| Object Storage | MinIO (S3-compatible) | Data Lake local, portável para Amazon S3 em produção |
| Orquestração | Apache Airflow | Agendamento, dependências, retentativa, observabilidade integrada |
| Format analítico | Delta Lake | ACID, MERGE incremental, time-travel, schema enforcement |
| Camadas de dados | Medalhão (4 camadas) | Separação clara de responsabilidades, reuso, qualidade progressiva |
| Modelagem Gold | Dimensional (Kimball) | Fatos e dimensões prontos para BI, performance de consultas OLAP |
| Histórico de dimensões | SCD Tipo 2 | Preserva versões de atributos ao longo do tempo para análise temporal |
| Carga incremental | `updated_at` + checkpoints | Reprocessa apenas dados alterados, eficiente e escalável |
| Containerização | Docker Compose | Ambiente reproducível, fácil setup para desenvolvimento e demo |
